# 🌊 FloodEdgeNet: Google Colab Benchmarking & Reproducibility Suite
### Khung thực nghiệm chuẩn Q1 ISI: Huấn luyện 5 Seeds, QAT INT8, Xuất ONNX, Đánh giá In-Domain, Cross-Domain & External Test (FloodVN)

**Ưu điểm nổi bật:**
- ⚙️ **Bảng điều khiển 1 chạm (Cell 2):** Chỉ cần đổi `MODEL_NAME`, toàn bộ pipeline tự động thích ứng với mô hình mới.
- 🧠 **Tự động nhận diện Pretrained:** Tự động kích hoạt ImageNet pretrained weights cho các mô hình nặng (như `deeplabv3` với ResNet-50 backbone), giữ train from scratch cho các mô hình Edge AI nhẹ (`flood_edgenet`, `fast-scnn`, v.v.). Bạn không cần sửa bất kỳ cờ lệnh nào!
- 🎲 **Đa Seed (5 Seeds độc lập):** Đảm bảo chuẩn nghiên cứu quốc tế Q1 ISI (báo cáo Mean ± Std và sinh bảng LaTeX tự động).
- 🔬 **Đánh giá toàn diện 3 miền:** In-Domain (cùng miền) + Cross-Domain (khác miền) + External Test (tập độc lập FloodVN).
- ⚡ **Tối ưu hóa Edge AI:** Lượng tử hóa QAT INT8 và xuất ONNX chuẩn hóa cho vi điều khiển MCU.

## 🛠️ Bước 1: Kiểm tra GPU & Cài đặt Thư viện

In [ ]:
# 1. Kiểm tra cấu hình phần cứng GPU Colab
!nvidia-smi

In [ ]:
# 2. Clone repository & Cài đặt dependencies (Chống lỗi khi chạy lại nhiều lần)
import os
if not os.path.exists('/content/FloodEdgeNet'):
    !git clone https://github.com/tuannh1905/FloodEdgeNet.git /content/FloodEdgeNet
    %cd /content/FloodEdgeNet
else:
    %cd /content/FloodEdgeNet
    !git pull

!pip install -q -r requirements.txt
!pip install -q onnx onnxscript gdown


## ⚙️ Bước 2: BẢNG ĐIỀU KHIỂN CẤU HÌNH TRUNG TÂM
👉 **Bạn chỉ cần thay đổi tên mô hình hoặc siêu tham số tại đây.** Toàn bộ quy trình bên dưới sẽ tự động cập nhật và tự động kích hoạt pretrained weights nếu là mô hình nặng.

In [ ]:
import os

# =========================================================================
# 1. CHỌN MÔ HÌNH THỰC NGHIỆM (Chỉ cần đổi tên model tại đây!)
#    Các mô hình có sẵn:
#    - Đề xuất: 'flood_edgenet'
#    - Lightweight Baselines: 'fast-scnn', 'espnetv2', 'stdc1', 'dabnet', 'edanet', 'enet', 'ulite'
#    - Standard Baselines: 'unet', 'segnet', 'deeplabv3', 'segformer'
#    - Ablation Models: 'ablation2_ECA', 'ablation3_CBAM', 'ablation4_NoAux', ...
# =========================================================================
MODEL_NAME  = "flood_edgenet"

# =========================================================================
# 2. TỰ ĐỘNG PHÁT HIỆN YÊU CẦU PRETRAINED WEIGHTS (Không cần can thiệp thủ công)
# =========================================================================
AUTO_PRETRAINED_MODELS = {'deeplabv3'}
IS_PRETRAINED = MODEL_NAME.lower() in AUTO_PRETRAINED_MODELS
PRETRAINED_FLAG = "--pretrained" if IS_PRETRAINED else ""

# =========================================================================
# 3. SIÊU THAM SỐ HUẤN LUYỆN (CHUẨN THEO YÊU CẦU)
# =========================================================================
IMAGE_SIZE  = 128             # Kích thước ảnh chuẩn MCU: 128x128
BATCH_SIZE  = 4               # Batch size = 4
EPOCHS      = 100             # Số epoch huấn luyện: 100
LR          = 0.001           # Learning rate: 0.001
LOSS_NAME   = "lovasz_focal"  # Hàm mất mát tối ưu: Lovasz + Focal Loss
NUM_CLASSES = 1               # Bài toán phân vùng lũ nhị phân (Binary 1-Class)

# =========================================================================
# 4. CHẾ ĐỘ MULTI-SEED (CHO BÀI BÁO Q1 ISI)
#    MULTISEED = True  -> Chạy đầy đủ 5 seeds [42, 123, 456, 789, 2024], tính Mean ± Std
#    MULTISEED = False -> Chạy 1 seed đại diện (42) để kiểm thử nhanh quy trình
# =========================================================================
MULTISEED   = True
SEEDS       = [42, 123, 456, 789, 2024]
REP_SEED    = 42              # Seed đại diện dùng cho QAT INT8 và ONNX export

# =========================================================================
# 5. THAM SỐ QUANTIZATION-AWARE TRAINING (QAT INT8)
# =========================================================================
QAT_EPOCHS  = 15
QAT_LR      = 1e-4

# Hàm hỗ trợ tự động định vị checkpoint tối ưu
def get_checkpoint_path(model, loss, dataset, seed, is_multiseed):
    clean_loss = loss.lower().replace('-', '_')
    candidates = [
        f"outputs/seed_{seed}/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/seed_{seed}/{model}_{loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{loss}_{dataset}_s{seed}.pth"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0] if is_multiseed else candidates[1]

print("=" * 75)
print(f"🚀 ĐÃ KHỞI TẠO CẤU HÌNH THỰC NGHIỆM:")
print(f"   - Mô hình           : {MODEL_NAME}")
if IS_PRETRAINED:
    print(f"   - Chế độ Pretrained : BẬT TỰ ĐỘNG (ImageNet Weights cho backbone ResNet-50)")
else:
    print(f"   - Chế độ Pretrained : TẮT (Edge AI Model nhẹ: Huấn luyện From Scratch độc lập)")
print(f"   - Kích thước ảnh    : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   - Batch Size / Epoch: {BATCH_SIZE} / {EPOCHS}")
print(f"   - Learning Rate     : {LR}")
print(f"   - Loss Function     : {LOSS_NAME}")
print(f"   - Multi-seed Mode   : {MULTISEED} (Seeds: {SEEDS if MULTISEED else [REP_SEED]})")
print("=" * 75)

---
## 🗂️ PHẦN A: HUẤN LUYỆN TRÊN DATASET 1 — `FloodKaggle`
- **Tự động tải dữ liệu:** Khi truyền `--download`
- **Tự động kiểm thử In-Domain:** Ngay sau khi kết thúc epoch cuối cùng, script sẽ tự nạp checkpoint tốt nhất và đánh giá trên tập Test của FloodKaggle, tính Mean ± Std qua 5 seeds.

In [ ]:
# A1. Huấn luyện FloodKaggle (Tự động Test In-Domain sau khi train xong)
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py \
      --dataset floodkaggle \
      --model {MODEL_NAME} \
      --num_classes {NUM_CLASSES} \
      --loss {LOSS_NAME} \
      --size {IMAGE_SIZE} \
      --epochs {EPOCHS} \
      --batch_size {BATCH_SIZE} \
      --lr {LR} \
      --multiseed \
      --seeds {seeds_str} \
      {PRETRAINED_FLAG} \
      --download
else:
    !python main.py \
      --dataset floodkaggle \
      --model {MODEL_NAME} \
      --num_classes {NUM_CLASSES} \
      --loss {LOSS_NAME} \
      --size {IMAGE_SIZE} \
      --epochs {EPOCHS} \
      --batch_size {BATCH_SIZE} \
      --lr {LR} \
      --seed {REP_SEED} \
      {PRETRAINED_FLAG} \
      --download

In [ ]:
# A2. Lượng tử hóa QAT INT8 trên FloodKaggle
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
print(f"[*] Checkpoint đầu vào cho QAT: {weights_fk}")

!python qat.py \
  --weights {weights_fk} \
  --model {MODEL_NAME} \
  --dataset floodkaggle \
  --num_classes {NUM_CLASSES} \
  --loss {LOSS_NAME} \
  --size {IMAGE_SIZE} \
  --epochs {QAT_EPOCHS} \
  --lr {QAT_LR} \
  --optimizer sgd

In [ ]:
# A3. Xuất mô hình sang ONNX deployable (cho vi điều khiển / Edge)
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python export.py \
  --model {MODEL_NAME} \
  --weights {weights_fk} \
  --size {IMAGE_SIZE} \
  --num_classes {NUM_CLASSES} \
  --output outputs/{MODEL_NAME}_floodkaggle.onnx

---
## 🏙️ PHẦN B: HUẤN LUYỆN TRÊN DATASET 2 — `AIFloodSense`
- Chuẩn hóa nhị phân 1-Class: Pixel 255 là vùng ngập lũ, còn lại là nền.
- Tự động đánh giá In-Domain trên tập Test của AIFloodSense ngay sau khi train xong.

In [ ]:
# B1. Huấn luyện AIFloodSense (Tự động Test In-Domain sau khi train xong)
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py \
      --dataset aifloodsense \
      --model {MODEL_NAME} \
      --num_classes {NUM_CLASSES} \
      --loss {LOSS_NAME} \
      --size {IMAGE_SIZE} \
      --epochs {EPOCHS} \
      --batch_size {BATCH_SIZE} \
      --lr {LR} \
      --multiseed \
      --seeds {seeds_str} \
      {PRETRAINED_FLAG} \
      --download
else:
    !python main.py \
      --dataset aifloodsense \
      --model {MODEL_NAME} \
      --num_classes {NUM_CLASSES} \
      --loss {LOSS_NAME} \
      --size {IMAGE_SIZE} \
      --epochs {EPOCHS} \
      --batch_size {BATCH_SIZE} \
      --lr {LR} \
      --seed {REP_SEED} \
      {PRETRAINED_FLAG} \
      --download

In [ ]:
# B2. Lượng tử hóa QAT INT8 trên AIFloodSense
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
print(f"[*] Checkpoint đầu vào cho QAT: {weights_afs}")

!python qat.py \
  --weights {weights_afs} \
  --model {MODEL_NAME} \
  --dataset aifloodsense \
  --num_classes {NUM_CLASSES} \
  --loss {LOSS_NAME} \
  --size {IMAGE_SIZE} \
  --epochs {QAT_EPOCHS} \
  --lr {QAT_LR} \
  --optimizer sgd

In [ ]:
# B3. Xuất mô hình sang ONNX deployable
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python export.py \
  --model {MODEL_NAME} \
  --weights {weights_afs} \
  --size {IMAGE_SIZE} \
  --num_classes {NUM_CLASSES} \
  --output outputs/{MODEL_NAME}_aifloodsense.onnx

---
## 🌐 PHẦN C: ĐÁNH GIÁ TỔNG QUÁT HÓA TOÀN DIỆN (IN-DOMAIN, CROSS-DOMAIN & EXTERNAL TEST)
Chạy đánh giá đồng thời trên cả 3 miền dữ liệu:
1. **In-Domain Test:** Đánh giá lại mô hình trên cùng tập test của nguồn đào tạo.
2. **Cross-Domain Test:** Đánh giá Zero-shot chuyển miền giữa `FloodKaggle` và `AIFloodSense`.
3. **External Test:** Đánh giá trên tập độc lập hoàn toàn **FloodVN** (tập ảnh `.PNG` và `.json` polygon từ link Google Drive mới).

In [ ]:
# C1. Đánh giá toàn diện với mô hình huấn luyện từ FloodKaggle
# (Tự động tải FloodVN nếu chưa có trên Colab qua cờ --download)
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)

!python cross_dataset_eval.py \
  --model {MODEL_NAME} \
  --weights {weights_fk} \
  --source_dataset floodkaggle \
  --target_datasets floodkaggle aifloodsense floodvn \
  --size {IMAGE_SIZE} \
  --num_classes {NUM_CLASSES} \
  --download

In [ ]:
# C2. Đánh giá toàn diện với mô hình huấn luyện từ AIFloodSense
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)

!python cross_dataset_eval.py \
  --model {MODEL_NAME} \
  --weights {weights_afs} \
  --source_dataset aifloodsense \
  --target_datasets aifloodsense floodkaggle floodvn \
  --size {IMAGE_SIZE} \
  --num_classes {NUM_CLASSES} \
  --download

---
## 📦 BƯỚC 3: TỰ ĐỘNG ĐÓNG GÓI TẤT CẢ KẾT QUẢ & TẢI VỀ MÁY TÍNH
Cell này sẽ:
1. Thu gom toàn bộ checkpoints của 5 seeds (`outputs/seed_42`, `seed_123`, ...)
2. Thu gom toàn bộ mô hình lượng tử hóa INT8 QAT (`.pth`)
3. Thu gom toàn bộ mô hình ONNX (`.onnx`)
4. Thu gom toàn bộ các file JSON thống kê kết quả và mã LaTeX bài báo
5. Nén thành file zip duy nhất và tự động kích hoạt trình duyệt tải xuống!

In [ ]:
import os
import shutil
from google.colab import files

bundle_dir = f"{MODEL_NAME}_All_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)

output_dir = "outputs"
total_items = 0

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  [📁 Thư mục Seed] Đã gom: {item}/")
            total_items += 1
        elif os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"  [📄 File Artifact] Đã gom: {item}")
            total_items += 1

zip_name = f"{MODEL_NAME}_All_Results.zip"
shutil.make_archive(f"{MODEL_NAME}_All_Results", "zip", bundle_dir)
zip_size_mb = os.path.getsize(zip_name) / (1024.0 * 1024.0)

print("=" * 75)
print(f"📦 TỔNG HỢP HOÀN TẤT: {total_items} mục được đóng gói vào {zip_name}")
print(f"📁 Dung lượng file ZIP: {zip_size_mb:.2f} MB")
print(f"💾 File lưu tại: {os.path.abspath(zip_name)}")
print("⬇️ Đang kích hoạt tải xuống trình duyệt...")
print("=" * 75)

try:
    files.download(zip_name)
except Exception as e:
    print(f"[INFO] Trình duyệt có thể chặn pop-up tự động tải ({e}).")
    print(f"       Bạn có thể tải thủ công file '{zip_name}' ở thanh thư mục bên trái Colab.")
